
# 06_silver_market_price
-Bronze -> Silver for `market_price` (internal CSV, 209 rows).


In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F
from datetime import date

SOURCE_NAME = "market_price"
REQUIRED_COLS = ["ticker", "bar_timestamp", "open", "high", "low", "close", "volume"]
KEY_COLS = ["ticker", "bar_timestamp"]
COMPARE_COLS = ["open", "high", "low", "close", "volume"]
business_date_str = date.today().isoformat()

In [0]:
bronze_df = read_bronze(spark, SOURCE_NAME)
print(f"Bronze row count: {bronze_df.count()}")

Bronze row count: 205


In [0]:
typed_df = (
    bronze_df
    .withColumn("ticker", F.trim(F.col("ticker")))
    .withColumn("bar_timestamp", F.to_timestamp("bar_timestamp"))  # preserves offset info if source string had it
    .withColumn("open", F.col("open").cast("double"))
    .withColumn("high", F.col("high").cast("double"))
    .withColumn("low", F.col("low").cast("double"))
    .withColumn("close", F.col("close").cast("double"))
    .withColumn("volume", F.col("volume").cast("long"))
)

In [0]:
clean_df, null_rejects_df = split_on_required_nulls(typed_df, REQUIRED_COLS)
null_reject_count = null_rejects_df.count()
if null_reject_count > 0:
    write_quarantine(null_rejects_df, SOURCE_NAME)

In [0]:
deduped_df, duplicates_df, breaks_df = split_duplicates(clean_df, KEY_COLS, COMPARE_COLS)
dup_count = duplicates_df.count()
break_count = breaks_df.count()
if dup_count > 0:
    write_quarantine(duplicates_df, SOURCE_NAME)
if break_count > 0:
    write_quarantine(breaks_df.withColumn("reason_code", F.lit("PRICE_ATTRIBUTE_BREAK")), SOURCE_NAME)

In [0]:
write_silver(deduped_df, SOURCE_NAME)
print(f"Silver row count: {deduped_df.count()}")

Silver row count: 205


In [0]:
log_dq(spark, SOURCE_NAME, business_date_str, "null_required_field", bronze_df.count(), null_reject_count, "NULL_REQUIRED_FIELD")
log_dq(spark, SOURCE_NAME, business_date_str, "duplicate_record", clean_df.count(), dup_count, "DUPLICATE_RECORD")
log_dq(spark, SOURCE_NAME, business_date_str, "attribute_break", clean_df.count(), break_count, "PRICE_ATTRIBUTE_BREAK")

/home/spark-c7d8a0e7-eef0-49cc-b0c8-f6/.ipykernel/71/command-5696143635338729-2886423099:181: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


In [0]:
bronze_count = bronze_df.count()
silver_count = deduped_df.count()
quarantined_count = null_reject_count + dup_count
assert bronze_count == silver_count + quarantined_count, (
    f"Row count mismatch: bronze={bronze_count}, silver={silver_count}, quarantined={quarantined_count}"
)
print(f"OK: bronze={bronze_count} = silver={silver_count} + quarantined={quarantined_count}")

OK: bronze=205 = silver=205 + quarantined=0
